# Oblig 2

- All exercises should be solved by splitting the dataset into 60% training, 20% validation,
and 20% test, using a stratified split so that the class ratio is preserved across
all three sets.

- Use the training set for 5-fold stratified cross-validation, the validation
set for model selection and hyperparameter tuning, and reserve the test set strictly
for final evaluation.

- For each experiment, report the mean and standard deviation
of accuracy (and, where specified, other metrics such as precision, recall, F1, or AUC)
among the folds. Given the class imbalance, accuracy alone is not sufficient; wherever it
is asked for, also report at least one imbalance-aware metric (e.g., balanced accuracy, F1,
or ROC-AUC).

In [38]:
import numpy as np
# import the dataset, and download for further use.
# Run this once to get the data

from ucimlrepo import fetch_ucirepo

def import_data() -> None:
    """
    Gets the dataset, and makes local csv
    """
    # fetch dataset
    adult = fetch_ucirepo(id=2)

    # data (as pandas dataframes)
    X = adult.data.features
    y = adult.data.targets

    X.to_csv("adult_features.csv", index=False)
    y.to_csv("adult_targets.csv", index=False)


In [39]:
# we now have the data locally as csv files
import pandas as pd

# import_data() # gets the dataset,

X = pd.read_csv("adult_features.csv") # features
y = pd.read_csv("adult_targets.csv")  # targets

print(X.shape)
print(X.dtypes)

(48842, 14)
age               int64
workclass           str
fnlwgt            int64
education           str
education-num     int64
marital-status      str
occupation          str
relationship        str
race                str
sex                 str
capital-gain      int64
capital-loss      int64
hours-per-week    int64
native-country      str
dtype: object


In [40]:
from sklearn.model_selection import train_test_split

def train_val_test_split(X, y):
    """
    Create a 60, 20, 20 split for train, test and validation
    :param X: The set containing the features
    :param y: The set containing the targets
    :return: X_train, X_val, X_test, y_train, y_val, y_test
    """
    # First split into training+validation (80%) and test (20%)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y  # Use stratify for classification
    )

    # Calculate the validation size relative to the train_val set
    # If test is 20%, validation is 20%, then validation is 20 / (100-20) = 20/80 = ~0.25
    val_size_relative = 0.20 / (1.0 - 0.20)

    # Split train_val into final training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=val_size_relative,
        random_state=42,
        stratify=y_train_val  # Use stratify again
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

# https://apxml.com/courses/introduction-to-neural-networks/chapter-2-data-preparation-neural-networks/data-splitting

## Data preparation

In [41]:
X.info()
# from .shape() we expect 48842 non-null values.
# we can see that there are missing values in the dataframe

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             48842 non-null  int64
 1   workclass       47879 non-null  str  
 2   fnlwgt          48842 non-null  int64
 3   education       48842 non-null  str  
 4   education-num   48842 non-null  int64
 5   marital-status  48842 non-null  str  
 6   occupation      47876 non-null  str  
 7   relationship    48842 non-null  str  
 8   race            48842 non-null  str  
 9   sex             48842 non-null  str  
 10  capital-gain    48842 non-null  int64
 11  capital-loss    48842 non-null  int64
 12  hours-per-week  48842 non-null  int64
 13  native-country  48568 non-null  str  
dtypes: int64(6), str(8)
memory usage: 5.2 MB


In [42]:
X.head(30)
# we can see that missing data is encoded as ?

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba
5,37,Private,284582,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States
6,49,Private,160187,9th,5,Married-spouse-absent,Other-service,Not-in-family,Black,Female,0,0,16,Jamaica
7,52,Self-emp-not-inc,209642,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,45,United-States
8,31,Private,45781,Masters,14,Never-married,Prof-specialty,Not-in-family,White,Female,14084,0,50,United-States
9,42,Private,159449,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,5178,0,40,United-States


In [43]:
# Q1.1

def missing_features(X)-> None:
    """
    Identify which features have missing data, and how many

    The spesific dataset used here encodes missing data as "?"
    Chec
    :param X: Feature set
    """

    X_copy = X # work on copy of the set

    X_copy[X_copy == '?'] = np.nan # replace ? with Nan, for string columns this is missing data
    print(X_copy.isnull().sum()) # count the amount of missing values

missing_features(X)

age                  0
workclass         2799
fnlwgt               0
education            0
education-num        0
marital-status       0
occupation        2809
relationship         0
race                 0
sex                  0
capital-gain         0
capital-loss         0
hours-per-week       0
native-country     857
dtype: int64


In [44]:
# Check correlation between missing occupation and workclass
def check_correlation(X, feature_1, feature_2, feature_3) -> None:
    """
    Given that both features have missing values. Checks the correlation between two features in a given dataset.
    :param X: dataset containing feature_1 and feature_2
    :param feature_1: first feature
    :param feature_2: second feature
    :return: None
    """
    X_relevant = X[[feature_1, feature_2, feature_3]]

    X_missing_corr= X_relevant.isnull().corr() # check correlation between missing values

    print(X_missing_corr)
    # the output will be dataframe (2x2)
    # Which tells if the two features are missing together
    # Value close to 1 means that the often are missing together
    # Value close to -1 means that they are mutually exclusive, ie feature_1 is missing and feature_2 exist
    # Close to 0 means no correlation, you can not tell anything about feature_2 given feature_1

check_correlation(X, "occupation", "workclass", "native-country")

                occupation  workclass  native-country
occupation        1.000000   0.998110       -0.002202
workclass         0.998110   1.000000       -0.002088
native-country   -0.002202  -0.002088        1.000000


From this we can see that the missing values are highly a missing occupation is highly associated with a missing workclass ($0.998 \approx 1 $). We can also see that missing native-country is unrealated to the other missing features.

In [45]:
missing_workclass = X['workclass'].isnull()
print(X.loc[missing_workclass, 'hours-per-week'].describe())

zero_hours_missing = (X.loc[X['workclass'].isnull(), 'hours-per-week'] == 0).sum()
print(f"Missing workclass AND 0 hours: {zero_hours_missing}")

count    2799.000000
mean       31.812433
std        15.070629
min         1.000000
25%        20.000000
50%        36.000000
75%        40.000000
max        99.000000
Name: hours-per-week, dtype: float64
Missing workclass AND 0 hours: 0


In [46]:
missing_workclass = X['workclass'].isnull()
print(X.loc[~missing_workclass, 'hours-per-week'].describe())

count    46043.000000
mean        40.945790
std         12.012468
min          1.000000
25%         40.000000
50%         40.000000
75%         45.000000
max         99.000000
Name: hours-per-week, dtype: float64


Based on [determine imputation](https://stats.stackexchange.com/questions/541337/how-should-i-determine-what-imputation-method-to-use) we conclude that the missing values are MAR (missing at random). Which means that "the probability the data are missing has something to do with variables". We have seen that the missing occupation correlates with missing workclass. There are also no row with missing workclass that is associated with 0 hours worked, which means that the correspondent is working. This suggests the missingness is better explained by other observed variables

We should not delete the rows with missing data as this would remove a non-random subset, it linked with occupation and correlated with a lower hrs/week. We decide on creating a missing category, this is because we are working with a categorical features, so mean imputation would not work. And because it is non-random it carries some information.

In [47]:
def replace_missing(X, feature_1, feature_2, feature_3) -> pd.DataFrame:
    """
    Replace missing data in rows with the column features_1-3, with a missing category
    :param X: Dataset
    :param feature_1: feature 1
    :param feature_2: feature 2
    :param feature_3: feature 3
    :return: New dataset with missing category
    """
    X['workclass'] = X[feature_1].fillna('Missing')
    X['occupation'] = X[feature_2].fillna('Missing')
    X['native-country'] = X[feature_3].fillna('Missing')

    return X

X_nonmissing = replace_missing(X, "occupation", "workclass", "native-country")

In [52]:
# Q1.2
def outliers_IQR(X, feature_list, verbose=True):
    """
    Detect outliers in numerical features using the IQR method.

    :param X: DataFrame containing the features
    :param feature_list: list of numerical column names to check
    :return: dict mapping feature name -> boolean mask (True = outlier)
    """
    outlier_masks = {}

    for col in feature_list:
        Q1 = X[col].quantile(0.25) # define Q1
        Q3 = X[col].quantile(0.75) # define Q3
        IQR = Q3 - Q1

        # create the bounds
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        mask = (X[col] < lower_bound) | (X[col] > upper_bound) # create a mask to check if a row is above or below the bound
        outlier_masks[col] = mask # apply the mask

        if verbose:
            print(f"{col}: outliers={mask.sum()} ({mask.mean()*100:.2f}%)")

    return outlier_masks

int_features = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
outlier_masks = outliers_IQR(X, int_features)


age: outliers=216 (0.44%)
fnlwgt: outliers=1453 (2.97%)
education-num: outliers=1794 (3.67%)
capital-gain: outliers=4035 (8.26%)
capital-loss: outliers=2282 (4.67%)
hours-per-week: outliers=13496 (27.63%)


In [49]:
# The dataset without missing values, split into 60/20/20 splits
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X_nonmissing, y) # todo: replace with the non-null data

# check sizes
print(f"Original dataset size: {len(X)}")
print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

#We now have a training set (60% of total data) which we can use StratifiedKFold on.

Original dataset size: 48842
Training set size: 29304
Validation set size: 9769
Test set size: 9769
